<a href="https://colab.research.google.com/github/iliaxant/Pattern_Recognition_Final_Project/blob/main/Final_Project_PR_58545.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Τελική Εργασία Εξαμήνου**

## Αναγνώριση Προτύπων - Ακαδημαϊκό έτος 2025-2026

## Ηλίας Ξανθόπουλος 58545

## GitHub Repo: https://github.com/iliaxant/Pattern_Recognition_Final_Project

---

1) Χειροκίνητο ανέβασμα του αρχείου *Final_Project_data.zip*.

2) Unziping του αρχείου *Final_Project_data.zip*:

In [5]:
import zipfile

zip_path = '/content/Final_Project_data.zip'
zip_ref = zipfile.ZipFile(zip_path, 'r')
zip_ref.extractall('/content')
zip_ref.close()

print("Data unzipped successfully to /content directory.")

Data unzipped successfully to /content directory.


3) Εγκατάσταση και φόρτωση των απαραίτητων βιβλιοθηκών.

In [6]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import random

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

4) Ορισμός συνάρτησης αρχικοποίησης όλων των χρησιμοποιούμενων random state machines για reproducibility.

In [7]:
def set_seed(seed=15):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

---

## **Μέρος 1ο – Ανίχνευση Αστοχίας σε Βιομηχανική Διαδικασία**

Φόρτωση των training και testing δεδομένων (*Training_data_manifacturing.csv* και *Test_data_manifacturing.csv* αρχεία αντίστοιχα). Ονομασία της κάθε στήλης (καθαρά προαιρετικό) και εκτύπωση χρήσιμων πληροφοριών για τα δεδομένα, όπως ό,τι αφορά τα missing values και την αναλογία των κλάσεων.

In [27]:
columns = [f"Feat {i}" for i in range(474)] + ["y"]

class_column = columns[-1]
data_columns = columns[0:474]

df_train = pd.read_csv("Training_data_manifacturing.csv", header=None, names=columns)
df_test = pd.read_csv("Test_data_manifacturing.csv", header=None, names=data_columns)



print('\n' + 25 * "=" + " Training Data " + 25 * "=" + '\n')
print(df_train.info())

print('\n' + 15 * "-" + " Missing Values Analysis " + 15 * "-" + '\n')

missing_per_feat = df_train.isnull().sum()
print(f'Missing Values List:')
print(missing_per_feat)

total_cells = df_train.size
total_missing = missing_per_feat.sum()
percent_missing = (total_missing / total_cells) * 100

print(f"\nTotal cells in set: {total_cells}")
print(f"Total missing values: {total_missing}")
print(f"Percentage of missing values: {(total_missing / total_cells) * 100:.2f}%")

print('\n' + 55 * "-")

print('\n' + 15 * "-" + " Class Info " + 15 * "-" + '\n')
print(df_train[class_column].value_counts())
print()
print(df_train[class_column].value_counts(normalize=True).map('{:.1%}'.format))
print('\n' + 42 * "-")
print('\n' + 65 * "=" + '\n\n')



print('\n' + 25 * "=" + " Testing Data " + 25 * "=" + '\n')
print(df_test.info())

print('\n' + 15 * "-" + " Missing Values Analysis " + 15 * "-" + '\n')

missing_per_feat = df_test.isnull().sum()
print(f'Missing Values List:')
print(missing_per_feat)

total_cells = df_test.size
total_missing = missing_per_feat.sum()
percent_missing = (total_missing / total_cells) * 100

print(f"\nTotal cells in set: {total_cells}")
print(f"Total missing values: {total_missing}")
print(f"Percentage of missing values: {(total_missing / total_cells) * 100:.2f}%")

print('\n' + 55 * "-")
print('\n' + 64 * "=" + '\n\n')



========================= Training Data =========================

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1254 entries, 0 to 1253
Columns: 475 entries, Feat 0 to y
dtypes: float64(474), int64(1)
memory usage: 4.5 MB
None

--------------- Missing Values Analysis ---------------

Missing Values List:
Feat 0      3
Feat 1      5
Feat 2      9
Feat 3      9
Feat 4      9
           ..
Feat 470    1
Feat 471    1
Feat 472    1
Feat 473    1
y           0
Length: 475, dtype: int64

Total cells in set: 595650
Total missing values: 32918
Percentage of missing values: 5.53%

-------------------------------------------------------

--------------- Class Info ---------------

y
0    1170
1      84
Name: count, dtype: int64

y
0    93.3%
1     6.7%
Name: proportion, dtype: object

------------------------------------------




========================= Testing Data =========================

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 313 entries, 0 to 312
Columns: 474 entries, F

Παρατηρούνται δύο πολύ σημαντικά προβλήματα στα δεδομένα.

Το πρώτο είναι τα missing values. Τόσο στα training όσο και στα testing δεδομένα υπάρχει ένα μεγάλο ποσοστό κελιών το οποίο δεν έχει καθόλου τιμή (~5.54%). Παρόλα αυτά, Όπως φαίνεται στις πληροφορίες για τα training δεδομένα, δεν υπάρχουν καθόλου NaN κελία στην στήλη των ετικετών, το οποίο σημαίνει ότι δεν χρειάζεται να απορριφθούν δείγματα, αλλά πρέπει οπωσδήποτε να επιλεχθεί μια στρατηγική αντιμετώπισης των missing values.

Το δεύτερο πρόβλημα αφορά τα training δεδομένα και είναι η ανισότητα κλάσεων. Όπως φαίνεται παραπάνω, η συντριπτική πλειοψηφία των δειγμάτων ανήκουν στην κλάση 0 και μόνο το 6.7% των training δεδομένων αντιστοιχούν στην κλάση 1. Επομένως πρέπει να εφαρμοστούν οι κατάλληλες τεχνικές για την διαχείρηση της ανισότητας κλάσεων.

Και τα δύο προβλήματα επιλύονται σε παρακάτω block.

**ΣΧΟΛΙΟ:** Στα παραπάνω μηνύματα για τα δεδομένα υπάρχει ένα discrepancy ως προς τον τύπο των δεδομένων.

Παρατηρούμε ότι στα training δεδομένα όλες οι στήλες είναι float64 εκτός από την στήλη των ετικετών που είναι int64. Στα testing δεδομένα που υπάρχουν τα ίδια χαρακτηριστικά αλλά δεν υπάρχει η στήλη ετικετών, πάλι βλέπουμε όλες τις στήλες να είναι float64, εκτός από μία που είναι int64.

Αυτό εν τέλει δεν είναι τυχαίο, διότι αυτόματα η ύπαρξη missing τιμών σε μία στήλη την μετατρέπει σε τύπο δεδομένων float64. Επομένως, στην περίπτωση των training δεδομένων, όλες οι στήλες χαρακτηριστικών τύπου int64 έχουν NaN τιμές, οπότε μετατρέπεται η στήλη σε float64, ενώ στα testing δεδομένα υπάρχει μια στήλη int64 που προηγουμένως είχε missing τιμές, αλλά τώρα δεν έχει, οπότε παραμένει int64.

Αυτό επιβεβαιώνεται τόσο από το παρακάτω code cell, όσο και από μια γρήγορη επισκόπηση των τιμών του αρχείου *Test_data_manifacturing.csv*.




In [46]:
print("*** Ignore 'dtype' lines ***\n")

int_cols_test = df_test.select_dtypes(include=['int64']).columns
print(f"Columns of int64 data type: {(list(int_cols_test))}")

print(f"\n--- Info for '{int_cols_test}' on Train data frame ---")
print(f"Data type: {df_train[int_cols_test].dtypes}")
print(f"Num of missing values: {df_train[int_cols_test].isnull().sum()}")

print(f"\n--- Info for '{int_cols_test}' on Test data frame ---")
print(f"Data type: {df_test[int_cols_test].dtypes}")
print(f"Num of missing values: {df_test[int_cols_test].isnull().sum()}")

*** Ignore 'dtype' lines ***

Columns of int64 data type: ['Feat 128']

--- Info for 'Index(['Feat 128'], dtype='object')' on Train data frame ---
Data type: Feat 128    float64
dtype: object
Num of missing values: Feat 128    5
dtype: int64

--- Info for 'Index(['Feat 128'], dtype='object')' on Test data frame ---
Data type: Feat 128    int64
dtype: object
Num of missing values: Feat 128    0
dtype: int64


### **a.**

### **b.**

### **c.**

---

## **Μέρος 2ο – Ταξινόμηση σε Υπερφασματική Εικόνα**

### **a.**

### **b.**

### **c.**